# RoadVision AI - Dataset Cleaning and Remediation

Produces an immutable, sanitized dataset copy under `roadvision_cleaned`. Actions include:
1. Strict byte-level deduplication (SHA-256) with split leakage prevention.
2. Removal of zero-area/degenerate YOLO lines without modifying valid sibling rows.
3. Explicit accounting and categorization of empty label files (true background vs. invalidated).
4. Detailed auditing preserved across CSV reports.

In [7]:
import os
import math
import shutil
import hashlib
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
from PIL import Image, UnidentifiedImageError
import yaml

DEFAULT_CACHE = Path.home() / ".cache" / "kagglehub" / "datasets" / "ashfakyeafi" / "road-vehicle-images-dataset" / "versions" / "2"
DATASET_VERSION_ROOT = Path(os.getenv("ROADVISION_DATASET_VERSION_ROOT", DEFAULT_CACHE))
ORIGINAL_ROOT = Path(os.getenv("ROADVISION_ORIGINAL_ROOT", DATASET_VERSION_ROOT / "trafic_data"))
CLEANED_ROOT = Path(os.getenv("ROADVISION_CLEANED_ROOT", DATASET_VERSION_ROOT / "roadvision_cleaned"))
REPORT_DIR = CLEANED_ROOT / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
SPLITS = ["train", "valid", "test"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

print("Original Root:", ORIGINAL_ROOT.resolve())
print("Cleaned Root :", CLEANED_ROOT.resolve())
assert ORIGINAL_ROOT.is_dir(), f"Original root missing: {ORIGINAL_ROOT}"
assert CLEANED_ROOT.resolve() != ORIGINAL_ROOT.resolve(), "Safety stop: CLEANED_ROOT matches ORIGINAL_ROOT!"

Original Root: C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\trafic_data
Cleaned Root : C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\roadvision_cleaned


### 1. Load Original YAML Schema

In [8]:
yaml_path = ORIGINAL_ROOT / "data_1.yaml"
with yaml_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f) or {}

names_raw = config.get("names", [])
if isinstance(names_raw, dict):
    class_names = [str(names_raw[k]) for k in sorted(names_raw, key=lambda x: int(x))]
elif isinstance(names_raw, list):
    class_names = [str(n) for n in names_raw]
nc = int(config.get("nc", len(class_names)))

print(f"Target Classes ({nc}):", class_names)

Target Classes (21): ['ambulance', 'army vehicle', 'auto rickshaw', 'bicycle', 'bus', 'car', 'garbagevan', 'human hauler', 'minibus', 'minivan', 'motorbike', 'pickup', 'policecar', 'rickshaw', 'scooter', 'suv', 'taxi', 'three wheelers -CNG-', 'truck', 'van', 'wheelbarrow']


### 2. Verify Image Integrity & Deduplicate Safely

In [9]:
image_integrity_records = []
hash_map = defaultdict(list)

for split in SPLITS:
    img_dir = ORIGINAL_ROOT / split / "images"
    if not img_dir.is_dir():
        continue
    for img_path in sorted(img_dir.iterdir()):
        if not img_path.is_file() or img_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        status, err, digest = "ok", "", None
        try:
            with Image.open(img_path) as im:
                im.verify()
            with Image.open(img_path) as im:
                im.load()
                if im.width <= 0 or im.height <= 0:
                    raise ValueError("Zero image dimension")
            digest = hashlib.sha256(img_path.read_bytes()).hexdigest()
            hash_map[digest].append((split, img_path))
        except (UnidentifiedImageError, OSError, ValueError) as ex:
            status, err = "corrupted", str(ex)
        image_integrity_records.append({
            "split": split, "image": str(img_path), "filename": img_path.name,
            "status": status, "error": err, "sha256": digest
        })

integrity_df = pd.DataFrame(image_integrity_records)
corrupted_df = integrity_df[integrity_df["status"] == "corrupted"].copy()

# Evaluate SHA-256 Duplicates and establish explicit retention policy
duplicate_decision_rows = []
images_to_exclude = set()

for digest, entries in hash_map.items():
    if len(entries) <= 1:
        continue
    # Deterministic selection: keep the first lexicographical split/filename as primary
    sorted_entries = sorted(entries, key=lambda x: (x[0], x[1].name))
    primary_split, primary_img = sorted_entries[0]
    primary_lbl = ORIGINAL_ROOT / primary_split / "labels" / f"{primary_img.stem}.txt"
    primary_content = primary_lbl.read_text(encoding="utf-8", errors="replace").strip() if primary_lbl.is_file() else ""

    duplicate_decision_rows.append({
        "sha256": digest, "split": primary_split, "image": str(primary_img),
        "role": "primary", "decision": "retained", "reason": "Deterministic lexicographical primary instance"
    })

    for sec_split, sec_img in sorted_entries[1:]:
        sec_lbl = ORIGINAL_ROOT / sec_split / "labels" / f"{sec_img.stem}.txt"
        sec_content = sec_lbl.read_text(encoding="utf-8", errors="replace").strip() if sec_lbl.is_file() else ""
        
        if sec_content == primary_content:
            images_to_exclude.add(str(sec_img))
            duplicate_decision_rows.append({
                "sha256": digest, "split": sec_split, "image": str(sec_img),
                "role": "duplicate", "decision": "excluded",
                "reason": "Byte-identical image and identical annotation content to primary"
            })
        else:
            # Differing annotations: retain under alert to prevent silent data loss
            duplicate_decision_rows.append({
                "sha256": digest, "split": sec_split, "image": str(sec_img),
                "role": "duplicate", "decision": "retained",
                "reason": "Annotations differ from primary instance; retained for manual inspection"
            })

duplicate_decisions_df = pd.DataFrame(duplicate_decision_rows)
print(f"Images evaluated: {len(integrity_df)}")
print(f"Corrupted images: {len(corrupted_df)}")
print(f"Duplicate decision entries: {len(duplicate_decisions_df)}")
display(duplicate_decisions_df)

Images evaluated: 3004
Corrupted images: 0
Duplicate decision entries: 2


,sha256,split,image,role,decision,reason
0,2ee1490b44e2afebab03e55838fe623aa31d99074bbf06...,train,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,primary,retained,Deterministic lexicographical primary instance
1,2ee1490b44e2afebab03e55838fe623aa31d99074bbf06...,train,C:\Users\USER\.cache\kagglehub\datasets\ashfak...,duplicate,excluded,Byte-identical image and identical annotation ...


### 3. Filter Degenerate YOLO Bounding Boxes Safely

In [10]:
def validate_yolo_line(line, num_classes):
    parts = line.strip().split()
    if len(parts) != 5:
        return False, "malformed_column_count", None, None
    try:
        cid_f = float(parts[0])
        if not math.isfinite(cid_f) or not cid_f.is_integer():
            return False, "class_id_not_integer", None, None
        class_id = int(cid_f)
        coords = [float(v) for v in parts[1:]]
    except ValueError:
        return False, "non_numeric_entry", None, None
    if not all(math.isfinite(v) for v in coords):
        return False, "non_finite_entry", None, class_id
    if not (0 <= class_id < num_classes):
        return False, "class_id_out_of_range", None, class_id
    xc, yc, w, h = coords
    if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0):
        return False, "center_out_of_bounds", None, class_id
    if not (0.0 < w <= 1.0 and 0.0 < h <= 1.0):
        return False, "non_positive_or_oversized_box", None, class_id
    x_min, x_max = xc - w / 2.0, xc + w / 2.0
    y_min, y_max = yc - h / 2.0, yc + h / 2.0
    if x_min < 0.0 or x_max > 1.0 or y_min < 0.0 or y_max > 1.0:
        return False, "box_exceeds_image_bounds", None, class_id
    clean_line = f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"
    return True, "", clean_line, class_id

cleaned_labels_by_stem = {}
invalid_lines_log = []
empty_labels_audit = []
cleaned_class_counts = Counter()

for split in SPLITS:
    lbl_dir = ORIGINAL_ROOT / split / "labels"
    if not lbl_dir.is_dir():
        continue
    for lbl_path in sorted(lbl_dir.glob("*.txt")):
        stem = lbl_path.stem
        raw_lines = [l.strip() for l in lbl_path.read_text(encoding="utf-8", errors="replace").splitlines() if l.strip()]
        valid_lines = []
        
        if not raw_lines:
            empty_labels_audit.append({
                "split": split, "label_file": lbl_path.name,
                "category": "legitimate_background", "original_box_count": 0, "valid_box_count": 0,
                "treatment": "retained_as_empty_background_label"
            })
        else:
            for line_no, raw_line in enumerate(raw_lines, start=1):
                ok, reason, sanitized_line, cid = validate_yolo_line(raw_line, nc)
                if ok:
                    valid_lines.append(sanitized_line)
                    cleaned_class_counts[cid] += 1
                else:
                    invalid_lines_log.append({
                        "split": split, "label_file": lbl_path.name, "line": line_no,
                        "reason": reason, "raw_content": raw_line, "class_id": cid
                    })
            if len(valid_lines) == 0:
                empty_labels_audit.append({
                    "split": split, "label_file": lbl_path.name,
                    "category": "invalidated_annotations", "original_box_count": len(raw_lines), "valid_box_count": 0,
                    "treatment": "retained_as_background_zero_annotation_file"
                })
        cleaned_labels_by_stem[(split, stem)] = valid_lines

invalid_lines_df = pd.DataFrame(invalid_lines_log)
empty_labels_df = pd.DataFrame(empty_labels_audit)
print(f"Removed invalid annotation lines: {len(invalid_lines_df)}")
print(f"Total empty label files: {len(empty_labels_df)}")
display(invalid_lines_df.head(25))
display(empty_labels_df)

Removed invalid annotation lines: 24
Total empty label files: 4


,split,label_file,line,reason,raw_content,class_id
0,valid,Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fff...,1,non_positive_or_oversized_box,9 0 0 0 0,9
1,valid,Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fff...,2,non_positive_or_oversized_box,15 0 0 0 0,15
2,valid,Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fff...,3,non_positive_or_oversized_box,13 0 0 0 0,13
3,valid,Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fff...,4,non_positive_or_oversized_box,13 0 0 0 0,13
4,valid,Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fff...,5,non_positive_or_oversized_box,13 0 0 0 0,13
5,valid,Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fff...,6,non_positive_or_oversized_box,13 0 0 0 0,13
6,valid,Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fff...,7,non_positive_or_oversized_box,13 0 0 0 0,13
7,valid,Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fff...,8,non_positive_or_oversized_box,13 0 0 0 0,13
8,valid,Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fff...,9,non_positive_or_oversized_box,13 0 0 0 0,13
9,valid,Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fff...,10,non_positive_or_oversized_box,13 0 0 0 0,13


,split,label_file,category,original_box_count,valid_box_count,treatment
0,train,01_jpg.rf.8d8a2f0f90d5b83893cd252acd832c93.txt,legitimate_background,0,0,retained_as_empty_background_label
1,train,231_jpg.rf.42292592b7fed42abf94d9624eda3fa5.txt,legitimate_background,0,0,retained_as_empty_background_label
2,valid,Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fff...,invalidated_annotations,14,0,retained_as_background_zero_annotation_file
3,valid,Pias--360-_PNG.rf.8405b0e44009a9300e0a1100ccf7...,invalidated_annotations,10,0,retained_as_background_zero_annotation_file


### 4. Build Cleaned Output Structure

In [11]:
cleaning_actions_log = []
corrupted_set = set(corrupted_df["image"].tolist())

for split in SPLITS:
    src_img_dir = ORIGINAL_ROOT / split / "images"
    src_lbl_dir = ORIGINAL_ROOT / split / "labels"
    dst_img_dir = CLEANED_ROOT / split / "images"
    dst_lbl_dir = CLEANED_ROOT / split / "labels"
    
    if not src_img_dir.is_dir():
        continue
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)
    
    for img_path in sorted(src_img_dir.iterdir()):
        if not img_path.is_file() or img_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        str_img = str(img_path)
        if str_img in corrupted_set:
            cleaning_actions_log.append({"split": split, "file": img_path.name, "action": "excluded_corrupted_image"})
            continue
        if str_img in images_to_exclude:
            cleaning_actions_log.append({"split": split, "file": img_path.name, "action": "excluded_duplicate_image"})
            continue
        
        # Copy image
        shutil.copy2(img_path, dst_img_dir / img_path.name)
        
        # Write cleaned label
        lbl_name = f"{img_path.stem}.txt"
        lines_to_write = cleaned_labels_by_stem.get((split, img_path.stem), [])
        lbl_content = "\n".join(lines_to_write) + ("\n" if lines_to_write else "")
        (dst_lbl_dir / lbl_name).write_text(lbl_content, encoding="utf-8")
        
        cleaning_actions_log.append({"split": split, "file": img_path.name, "action": "copied_image_and_cleaned_label"})

actions_df = pd.DataFrame(cleaning_actions_log)
print("Cleaning execution complete. Summary of actions:")
display(actions_df["action"].value_counts().reset_index())

Cleaning execution complete. Summary of actions:


,action,count
0,copied_image_and_cleaned_label,3003
1,excluded_duplicate_image,1


### 5. Export Reports and Final Clean Distribution

In [13]:
integrity_df.to_csv(REPORT_DIR / "image_integrity.csv", index=False)
corrupted_df.to_csv(REPORT_DIR / "corrupted_images.csv", index=False)
duplicate_decisions_df.to_csv(REPORT_DIR / "duplicate_handling_decisions.csv", index=False)
invalid_lines_df.to_csv(REPORT_DIR / "invalid_annotation_lines.csv", index=False)
empty_labels_df.to_csv(REPORT_DIR / "empty_labels_audit.csv", index=False)
actions_df.to_csv(REPORT_DIR / "cleaning_actions.csv", index=False)

clean_dist_df = pd.DataFrame([
    {"class_id": i, "class_name": class_names[i], "valid_boxes_cleaned": cleaned_class_counts.get(i, 0)}
    for i in range(nc)
])
clean_dist_df.to_csv(REPORT_DIR / "cleaned_class_distribution.csv", index=False)

print(f"All cleaning audit reports saved to: {REPORT_DIR.resolve()}")
display(clean_dist_df)

All cleaning audit reports saved to: C:\Users\USER\.cache\kagglehub\datasets\ashfakyeafi\road-vehicle-images-dataset\versions\2\roadvision_cleaned\reports


,class_id,class_name,valid_boxes_cleaned
0,0,ambulance,70
1,1,army vehicle,43
2,2,auto rickshaw,372
3,3,bicycle,459
4,4,bus,3332
5,5,car,5474
6,6,garbagevan,3
7,7,human hauler,169
8,8,minibus,95
9,9,minivan,931
